In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Localizar o dataset no projeto ou nos Downloads
candidatos = [
    Path("data/raw/SAML-D.csv"),
    Path("FraudRadar/data/raw/SAML-D.csv"),
    Path.home() / "Downloads" / "archive" / "SAML-D.csv"
]

caminho = next(
    (p.resolve() for p in candidatos if p.is_file()),
    None
)

if caminho is None:
    raise FileNotFoundError(
        "Dataset não encontrado. Coloque o arquivo em "
        "data/raw/SAML-D.csv ou em Downloads/archive/SAML-D.csv."
    )

# Tipos de dados
tipos = {
    "Time": "string",
    "Date": "string",
    "Sender_account": "int64",
    "Receiver_account": "int64",
    "Amount": "float64",
    "Payment_currency": "category",
    "Received_currency": "category",
    "Sender_bank_location": "category",
    "Receiver_bank_location": "category",
    "Payment_type": "category",
    "Is_laundering": "int8",
    "Laundering_type": "category"
}

# Carregar o dataset
df = pd.read_csv(
    caminho,
    dtype=tipos,
    low_memory=False
)

df["timestamp"] = pd.to_datetime(
    df["Date"] + " " + df["Time"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

print("Arquivo carregado:", caminho)
print("Dimensões:", df.shape)
print("Memória utilizada:", round(
    df.memory_usage(deep=True).sum() / 1024**3, 2
), "GB")

In [ ]:
# 1. Tipos de dados e valores preenchidos
print("=== ESTRUTURA ===")
df.info()

# 2. Valores ausentes
print("\n=== VALORES AUSENTES ===")
display(df.isna().sum().to_frame("Ausentes"))

# 3. Quantidade de valores diferentes por coluna
print("\n=== VALORES ÚNICOS ===")
display(df.nunique().to_frame("Valores únicos"))

# 4. Distribuição da variável que queremos prever
print("\n=== IS_LAUNDERING ===")
display(df["Is_laundering"].value_counts(dropna=False))

display(
    (df["Is_laundering"].value_counts(normalize=True) * 100)
    .round(4)
    .to_frame("Percentual (%)")
)

# 5. Resumo dos valores das transações
print("\n=== VALORES DAS TRANSAÇÕES ===")
display(df["Amount"].describe())

# 6. Tipos de pagamento mais frequentes
print("\n=== TIPOS DE PAGAMENTO ===")
display(df["Payment_type"].value_counts().head(15))

# 7. Intervalo de datas da amostra
print("\n=== PERÍODO DOS DADOS ===")
print("Data inicial:", df["Date"].min())
print("Data final:", df["Date"].max())

In [ ]:
# 1. Verificar a relação entre a classe e a tipologia
print("=== CLASSE X TIPOLOGIA ===")

display(pd.crosstab(
    df["Is_laundering"],
    df["Laundering_type"]
))

# 2. Ver os valores das transações por classe
print("=== VALORES POR CLASSE ===")

display(
    df.groupby("Is_laundering")["Amount"]
    .describe()
    .round(2)
)

# 3. Conferir quantas linhas há por data
print("=== QUANTIDADE POR DATA ===")

display(df["Date"].value_counts().sort_index())

# 4. Verificar linhas duplicadas
print("=== DUPLICADAS ===")

print(df.duplicated().sum())

In [ ]:
import numpy as np
import pandas as pd

# Criar uma cópia para o primeiro experimento
df_modelo = df.copy()

# Transformar data e hora em um único campo temporal
df_modelo["timestamp"] = pd.to_datetime(
    df_modelo["Date"] + " " + df_modelo["Time"],
    errors="coerce"
)

# Criar características temporais
df_modelo["hora"] = df_modelo["timestamp"].dt.hour
df_modelo["dia_semana"] = df_modelo["timestamp"].dt.dayofweek

# Transformar o valor monetário para reduzir a influência de valores extremos
df_modelo["valor_log"] = np.log1p(df_modelo["Amount"])

# Definir as variáveis de entrada
colunas_numericas = ["valor_log", "hora", "dia_semana"]

colunas_categoricas = [
    "Payment_currency",
    "Received_currency",
    "Sender_bank_location",
    "Receiver_bank_location",
    "Payment_type"
]

colunas_modelo = colunas_numericas + colunas_categoricas

X = df_modelo[colunas_modelo]
y = df_modelo["Is_laundering"]

# Separar cronologicamente: último dia para teste
data_teste = df_modelo["Date"].max()
mascara_teste = df_modelo["Date"] == data_teste

X_treino = X.loc[~mascara_teste]
y_treino = y.loc[~mascara_teste]

X_teste = X.loc[mascara_teste]
y_teste = y.loc[mascara_teste]

# Conferir o resultado
print("Variáveis de entrada:")
print(X.columns.tolist())

print("\nData de teste:", data_teste)

print("\nTreino:", X_treino.shape)
print("Teste:", X_teste.shape)

print("\nClasses no treino:")
print(y_treino.value_counts())

print("\nClasses no teste:")
print(y_teste.value_counts())

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    average_precision_score,
    confusion_matrix
)

# Preparar as variáveis numéricas e categóricas
preprocessador = ColumnTransformer([
    ("numericas", StandardScaler(), colunas_numericas),
    ("categoricas", OneHotEncoder(handle_unknown="ignore"), colunas_categoricas)
])

# Criar o modelo
modelo = Pipeline([
    ("preprocessamento", preprocessador),
    ("classificador", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

# Treinar exclusivamente com os dados de treino
modelo.fit(X_treino, y_treino)

# Prever a classe e a probabilidade de suspeita
y_pred = modelo.predict(X_teste)
y_prob = modelo.predict_proba(X_teste)[:, 1]

# Avaliar o resultado
print("=== RESULTADO DA CLASSIFICAÇÃO ===")
print(classification_report(
    y_teste,
    y_pred,
    target_names=["Normal", "Suspeita"],
    zero_division=0
))

print("=== MATRIZ DE CONFUSÃO ===")
print(confusion_matrix(y_teste, y_pred))

print("=== PR-AUC (Average Precision) ===")
print(round(average_precision_score(y_teste, y_prob), 4))

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

resultados = []

for limiar in [0.5, 0.7, 0.8, 0.9, 0.95, 0.99]:
    previsoes = (y_prob >= limiar).astype(int)

    resultados.append({
        "Limiar": limiar,
        "Alertas": int(previsoes.sum()),
        "Verdadeiros positivos": int(
            ((previsoes == 1) & (y_teste == 1)).sum()
        ),
        "Falsos positivos": int(
            ((previsoes == 1) & (y_teste == 0)).sum()
        ),
        "Precision (%)": round(
            precision_score(y_teste, previsoes, zero_division=0) * 100, 2
        ),
        "Recall (%)": round(
            recall_score(y_teste, previsoes, zero_division=0) * 100, 2
        ),
        "F1 (%)": round(
            f1_score(y_teste, previsoes, zero_division=0) * 100, 2
        )
    })

display(pd.DataFrame(resultados))

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    classification_report,
    average_precision_score,
    confusion_matrix
)

# Preparar os dados para o segundo modelo
preprocessador_hgb = ColumnTransformer(
    transformers=[
        ("numericas", "passthrough", colunas_numericas),
        (
            "categoricas",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            colunas_categoricas
        )
    ],
    sparse_threshold=0
)

# Criar o classificador
modelo_hgb = Pipeline([
    ("preprocessamento", preprocessador_hgb),
    (
        "classificador",
        HistGradientBoostingClassifier(
            max_iter=100,
            learning_rate=0.1,
            max_leaf_nodes=31,
            class_weight="balanced",
            early_stopping=False,
            random_state=42
        )
    )
])

# Treinar
modelo_hgb.fit(X_treino, y_treino)

# Prever
y_pred_hgb = modelo_hgb.predict(X_teste)
y_prob_hgb = modelo_hgb.predict_proba(X_teste)[:, 1]

# Avaliar
print("=== RESULTADO: HISTGRADIENTBOOSTING ===")

print(classification_report(
    y_teste,
    y_pred_hgb,
    target_names=["Normal", "Suspeita"],
    zero_division=0
))

print("=== MATRIZ DE CONFUSÃO ===")
print(confusion_matrix(y_teste, y_pred_hgb))

print("=== PR-AUC (AVERAGE PRECISION) ===")
print(round(
    average_precision_score(y_teste, y_prob_hgb),
    4
))

In [ ]:
# Criar uma cópia e ordenar as transações no tempo
df_hist = df.copy()

df_hist["timestamp"] = pd.to_datetime(
    df_hist["Date"] + " " + df_hist["Time"]
)

df_hist = df_hist.sort_values(
    "timestamp", kind="stable"
).reset_index(drop=True)

# 1. Histórico de envios por remetente
envios = (
    df_hist.groupby(
        ["Sender_account", "timestamp"], as_index=False
    )
    .agg(
        envios_instante=("Amount", "size"),
        soma_valores=("Amount", "sum")
    )
    .sort_values(["Sender_account", "timestamp"])
)

grupo = envios.groupby("Sender_account", sort=False)

envios["envios_anteriores"] = (
    grupo["envios_instante"].cumsum()
    - envios["envios_instante"]
)

envios["soma_anterior"] = (
    grupo["soma_valores"].cumsum()
    - envios["soma_valores"]
)

envios["ultima_transacao"] = grupo["timestamp"].shift(1)

envios["media_valor_anterior"] = (
    envios["soma_anterior"]
    / envios["envios_anteriores"].replace(0, np.nan)
)

envios["segundos_desde_envio"] = (
    envios["timestamp"] - envios["ultima_transacao"]
).dt.total_seconds()

df_hist = df_hist.merge(
    envios[
        [
            "Sender_account",
            "timestamp",
            "envios_anteriores",
            "media_valor_anterior",
            "segundos_desde_envio"
        ]
    ],
    on=["Sender_account", "timestamp"],
    how="left",
    validate="many_to_one"
)

# 2. Histórico de recebimentos por destinatário
recebimentos = (
    df_hist.groupby(
        ["Receiver_account", "timestamp"], as_index=False
    )
    .size()
    .rename(columns={"size": "recebimentos_instante"})
    .sort_values(["Receiver_account", "timestamp"])
)

recebimentos["recebimentos_anteriores"] = (
    recebimentos.groupby(
        "Receiver_account", sort=False
    )["recebimentos_instante"].cumsum()
    - recebimentos["recebimentos_instante"]
)

df_hist = df_hist.merge(
    recebimentos[
        ["Receiver_account", "timestamp", "recebimentos_anteriores"]
    ],
    on=["Receiver_account", "timestamp"],
    how="left",
    validate="many_to_one"
)

# 3. Histórico da relação entre remetente e destinatário
pares = (
    df_hist.groupby(
        ["Sender_account", "Receiver_account", "timestamp"],
        as_index=False
    )
    .size()
    .rename(columns={"size": "transacoes_do_par"})
    .sort_values(
        ["Sender_account", "Receiver_account", "timestamp"]
    )
)

pares["interacoes_anteriores"] = (
    pares.groupby(
        ["Sender_account", "Receiver_account"], sort=False
    )["transacoes_do_par"].cumsum()
    - pares["transacoes_do_par"]
)

df_hist = df_hist.merge(
    pares[
        [
            "Sender_account",
            "Receiver_account",
            "timestamp",
            "interacoes_anteriores"
        ]
    ],
    on=["Sender_account", "Receiver_account", "timestamp"],
    how="left",
    validate="many_to_one"
)

df_hist["par_previamente_visto"] = (
    df_hist["interacoes_anteriores"] > 0
).astype(int)

# Tratar contas sem histórico anterior
colunas_historicas = [
    "envios_anteriores",
    "media_valor_anterior",
    "segundos_desde_envio",
    "recebimentos_anteriores"
]

df_hist["envios_anteriores"] = (
    df_hist["envios_anteriores"].fillna(0)
)

df_hist["media_valor_anterior"] = (
    df_hist["media_valor_anterior"].fillna(0)
)

df_hist["segundos_desde_envio"] = (
    df_hist["segundos_desde_envio"].fillna(-1)
)

df_hist["recebimentos_anteriores"] = (
    df_hist["recebimentos_anteriores"].fillna(0)
)

print("Dimensões:", df_hist.shape)

display(
    df_hist[
        [
            "timestamp",
            "Sender_account",
            "Amount",
            "envios_anteriores",
            "media_valor_anterior",
            "segundos_desde_envio",
            "Receiver_account",
            "recebimentos_anteriores",
            "par_previamente_visto",
            "Is_laundering"
        ]
    ].head(10)
)

In [ ]:
colunas_historicas = [
    "envios_anteriores",
    "media_valor_anterior",
    "segundos_desde_envio",
    "recebimentos_anteriores",
    "interacoes_anteriores",
    "par_previamente_visto"
]

print("=== RESUMO DAS VARIÁVEIS HISTÓRICAS ===")
display(df_hist[colunas_historicas].describe().round(2))

print("\n=== CONTAS COM HISTÓRICO ANTERIOR ===")
print(
    "Remetentes com envios anteriores:",
    (df_hist["envios_anteriores"] > 0).sum()
)

print(
    "Destinatários com recebimentos anteriores:",
    (df_hist["recebimentos_anteriores"] > 0).sum()
)

print(
    "Pares que já transacionaram antes:",
    (df_hist["par_previamente_visto"] == 1).sum()
)

print("\n=== EXEMPLOS COM HISTÓRICO ===")

display(
    df_hist.loc[
        (df_hist["envios_anteriores"] > 0)
        | (df_hist["recebimentos_anteriores"] > 0)
        | (df_hist["par_previamente_visto"] == 1),
        [
            "Date",
            "Time",
            "Sender_account",
            "Receiver_account",
            "Amount",
            "envios_anteriores",
            "media_valor_anterior",
            "segundos_desde_envio",
            "recebimentos_anteriores",
            "par_previamente_visto"
        ]
    ].head(10)
)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
import numpy as np
import pandas as pd

# 1. Criar as variáveis temporais e monetárias
df_hist["hora"] = df_hist["timestamp"].dt.hour
df_hist["dia_semana"] = df_hist["timestamp"].dt.day_of_week
df_hist["valor_log"] = np.log1p(df_hist["Amount"])

# 2. Definir as entradas, incluindo o histórico
numericas = [
    "valor_log",
    "hora",
    "dia_semana",
    "envios_anteriores",
    "media_valor_anterior",
    "segundos_desde_envio",
    "recebimentos_anteriores",
    "interacoes_anteriores"
]

categoricas = [
    "Payment_currency",
    "Received_currency",
    "Sender_bank_location",
    "Receiver_bank_location",
    "Payment_type"
]

X_hist = df_hist[numericas + categoricas]
y_hist = df_hist["Is_laundering"].astype(int)

# 3. Separar passado para treino e último dia para teste
data_teste = df_hist["Date"].max()
treino_mask = df_hist["Date"] < data_teste
teste_mask = df_hist["Date"] == data_teste

X_tr = X_hist.loc[treino_mask]
y_tr = y_hist.loc[treino_mask]
X_te = X_hist.loc[teste_mask]
y_te = y_hist.loc[teste_mask]

print("Data de teste:", data_teste)
print("Treino:", X_tr.shape, "| Suspeitas:", int(y_tr.sum()))
print("Teste:", X_te.shape, "| Suspeitas:", int(y_te.sum()))

# 4. Pré-processamento
preprocessador = ColumnTransformer([
    ("num", StandardScaler(), numericas),
    ("cat", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ), categoricas)
], sparse_threshold=0)

# 5. Criar os dois modelos
modelos = {
    "Regressão Logística": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=100,
        learning_rate=0.1,
        max_leaf_nodes=31,
        class_weight="balanced",
        early_stopping=False,
        random_state=42
    )
}

resultados = []

# 6. Treinar e avaliar
for nome, classificador in modelos.items():
    modelo = Pipeline([
        ("preprocessamento", preprocessador),
        ("classificador", classificador)
    ])

    modelo.fit(X_tr, y_tr)

    probabilidades = modelo.predict_proba(X_te)[:, 1]
    previsoes = (probabilidades >= 0.5).astype(int)

    resultados.append({
        "Modelo": nome,
        "PR-AUC": average_precision_score(y_te, probabilidades),
        "Precision (%)": precision_score(
            y_te, previsoes, zero_division=0
        ) * 100,
        "Recall (%)": recall_score(
            y_te, previsoes, zero_division=0
        ) * 100,
        "F1 (%)": f1_score(
            y_te, previsoes, zero_division=0
        ) * 100,
        "Alertas": int(previsoes.sum()),
        "Verdadeiros positivos": int(
            ((previsoes == 1) & (y_te.to_numpy() == 1)).sum()
        ),
        "Falsos positivos": int(
            ((previsoes == 1) & (y_te.to_numpy() == 0)).sum()
        )
    })

    print(f"\n=== {nome} ===")
    print("Matriz de confusão:")
    print(confusion_matrix(y_te, previsoes))

comparacao = pd.DataFrame(resultados).set_index("Modelo")

print("\n=== COMPARAÇÃO DOS MODELOS COM HISTÓRICO ===")
display(comparacao.round(4))

In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# ==================================================
# 1. DEFINIR AS VARIÁVEIS
# ==================================================

df_exp = df_hist.copy()

# Garantir as características temporais e monetárias
df_exp["hora"] = df_exp["timestamp"].dt.hour
df_exp["dia_semana"] = df_exp["timestamp"].dt.day_of_week
df_exp["valor_log"] = np.log1p(df_exp["Amount"])

# Variáveis disponíveis nos dois experimentos
numericas_base = [
    "valor_log",
    "hora",
    "dia_semana"
]

# Variáveis históricas adicionadas no experimento B
numericas_historico = numericas_base + [
    "envios_anteriores",
    "media_valor_anterior",
    "segundos_desde_envio",
    "recebimentos_anteriores",
    "interacoes_anteriores"
]

categoricas = [
    "Payment_currency",
    "Received_currency",
    "Sender_bank_location",
    "Receiver_bank_location",
    "Payment_type"
]

# Garantir datas em formato padronizado
df_exp["_data"] = pd.to_datetime(
    df_exp["Date"]
).dt.strftime("%Y-%m-%d")

# ==================================================
# 2. SEPARAÇÃO TEMPORAL
# ==================================================

treino_mask = df_exp["_data"] <= "2022-10-08"
validacao_mask = df_exp["_data"] == "2022-10-09"
teste_mask = df_exp["_data"] == "2022-10-10"

y = df_exp["Is_laundering"].astype(int)

y_treino = y.loc[treino_mask]
y_validacao = y.loc[validacao_mask]
y_teste = y.loc[teste_mask]

print("=== DISTRIBUIÇÃO DAS CLASSES ===")
print(
    df_exp.groupby("_data")["Is_laundering"]
    .agg(Total="count", Suspeitas="sum")
)

print("\nTreino:", len(y_treino),
      "| Suspeitas:", int(y_treino.sum()))
print("Validação:", len(y_validacao),
      "| Suspeitas:", int(y_validacao.sum()))
print("Teste:", len(y_teste),
      "| Suspeitas:", int(y_teste.sum()))

if min(y_treino.nunique(), y_validacao.nunique(), y_teste.nunique()) < 2:
    raise ValueError(
        "Uma das divisões não possui as duas classes. "
        "Precisamos ajustar os períodos antes de continuar."
    )

# ==================================================
# 3. DEFINIR OS QUATRO EXPERIMENTOS
# ==================================================

experimentos = [
    (
        "Logística - sem histórico",
        numericas_base,
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        )
    ),
    (
        "Logística - com histórico",
        numericas_historico,
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        )
    ),
    (
        "HistGradientBoosting - sem histórico",
        numericas_base,
        HistGradientBoostingClassifier(
            max_iter=100,
            learning_rate=0.1,
            max_leaf_nodes=31,
            class_weight="balanced",
            early_stopping=False,
            random_state=42
        )
    ),
    (
        "HistGradientBoosting - com histórico",
        numericas_historico,
        HistGradientBoostingClassifier(
            max_iter=100,
            learning_rate=0.1,
            max_leaf_nodes=31,
            class_weight="balanced",
            early_stopping=False,
            random_state=42
        )
    )
]

resultados = []

# ==================================================
# 4. TREINAR, VALIDAR E TESTAR
# ==================================================

for nome, numericas, classificador in experimentos:

    colunas = numericas + categoricas

    X_treino = df_exp.loc[treino_mask, colunas]
    X_validacao = df_exp.loc[validacao_mask, colunas]
    X_teste = df_exp.loc[teste_mask, colunas]

    preprocessador = ColumnTransformer(
        transformers=[
            ("numericas", StandardScaler(), numericas),
            (
                "categoricas",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ),
                categoricas
            )
        ],
        sparse_threshold=0
    )

    modelo = Pipeline([
        ("preprocessamento", preprocessador),
        ("classificador", classificador)
    ])

    modelo.fit(X_treino, y_treino)

    # A validação escolhe o limiar
    prob_validacao = modelo.predict_proba(
        X_validacao
    )[:, 1]

    precisao, recall, limiares = precision_recall_curve(
        y_validacao,
        prob_validacao
    )

    f1_validacao = (
        2 * precisao[:-1] * recall[:-1]
        / (precisao[:-1] + recall[:-1] + 1e-12)
    )

    melhor_indice = np.argmax(f1_validacao)
    limiar = limiares[melhor_indice]

    # O teste é usado somente para a avaliação final
    prob_teste = modelo.predict_proba(X_teste)[:, 1]
    pred_teste = (prob_teste >= limiar).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_teste,
        pred_teste,
        labels=[0, 1]
    ).ravel()

    resultados.append({
        "Modelo": nome,
        "Limiar escolhido": limiar,
        "AP validação": average_precision_score(
            y_validacao, prob_validacao
        ),
        "AP teste": average_precision_score(
            y_teste, prob_teste
        ),
        "Alertas teste": int(pred_teste.sum()),
        "Suspeitas encontradas": int(tp),
        "Falsos positivos": int(fp),
        "Precision (%)": 100 * precision_score(
            y_teste, pred_teste, zero_division=0
        ),
        "Recall (%)": 100 * recall_score(
            y_teste, pred_teste, zero_division=0
        ),
        "F1 (%)": 100 * f1_score(
            y_teste, pred_teste, zero_division=0
        )
    })

    print(f"\n=== {nome} ===")
    print("Matriz de confusão:")
    print(confusion_matrix(y_teste, pred_teste))

# ==================================================
# 5. COMPARAR RESULTADOS
# ==================================================

comparacao = pd.DataFrame(resultados).set_index("Modelo")

print("\n=== COMPARAÇÃO FINAL ===")
display(comparacao.round(4))

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score

# Os mesmos quatro experimentos, usando a divisão temporal atual
experimentos = [
    ("Logística - sem histórico", numericas_base,
     LogisticRegression(
         max_iter=1000, class_weight="balanced", random_state=42
     )),

    ("Logística - com histórico", numericas_historico,
     LogisticRegression(
         max_iter=1000, class_weight="balanced", random_state=42
     )),

    ("HistGradientBoosting - sem histórico", numericas_base,
     HistGradientBoostingClassifier(
         max_iter=100, learning_rate=0.1, max_leaf_nodes=31,
         class_weight="balanced", early_stopping=False, random_state=42
     )),

    ("HistGradientBoosting - com histórico", numericas_historico,
     HistGradientBoostingClassifier(
         max_iter=100, learning_rate=0.1, max_leaf_nodes=31,
         class_weight="balanced", early_stopping=False, random_state=42
     ))
]

# Preparar o alvo e o teste
y_teste_atual = df_exp.loc[teste_mask, "Is_laundering"].astype(int)
total_suspeitas = int(y_teste_atual.sum())

resultados_topk = []

for nome, numericas, classificador in experimentos:
    colunas = numericas + categoricas

    X_tr = df_exp.loc[treino_mask, colunas]
    X_te = df_exp.loc[teste_mask, colunas]
    y_tr = df_exp.loc[treino_mask, "Is_laundering"].astype(int)

    preprocessador = ColumnTransformer([
        ("num", StandardScaler(), numericas),
        ("cat", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ), categoricas)
    ], sparse_threshold=0)

    modelo = Pipeline([
        ("preprocessamento", preprocessador),
        ("classificador", classificador)
    ])

    modelo.fit(X_tr, y_tr)

    # Probabilidade de cada transação ser suspeita
    scores = modelo.predict_proba(X_te)[:, 1]

    # Ordenar da maior para a menor probabilidade
    ordem = scores.argsort()[::-1]
    classes_ordenadas = y_teste_atual.to_numpy()[ordem]

    for k in [25, 50, 100, 200, 300]:
        suspeitas_encontradas = int(classes_ordenadas[:k].sum())

        resultados_topk.append({
            "Modelo": nome,
            "AP teste": average_precision_score(
                y_teste_atual, scores
            ),
            "Top K": k,
            "Suspeitas encontradas": suspeitas_encontradas,
            "Precisão (%)": round(
                suspeitas_encontradas / k * 100, 2
            ),
            "Recall (%)": round(
                suspeitas_encontradas / total_suspeitas * 100, 2
            )
        })

tabela_topk = pd.DataFrame(resultados_topk)

display(
    tabela_topk.sort_values(["Top K", "Modelo"])
    .reset_index(drop=True)
)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score

# Separar treino e validação mantendo a ordem temporal
dados_treino = (
    df_exp.loc[treino_mask]
    .sort_values("timestamp")
    .copy()
)

dados_validacao = df_exp.loc[validacao_mask].copy()

colunas = numericas_historico + categoricas

X_validacao = dados_validacao[colunas]
y_validacao = dados_validacao["Is_laundering"].astype(int)

# Quantidades de dados utilizadas em cada treinamento
fracoes = [0.2, 0.4, 0.6, 0.8, 1.0]

resultados_curva = []

for fracao in fracoes:

    # Usar uma porção cronológica dos dados de treino
    quantidade = int(len(dados_treino) * fracao)
    amostra = dados_treino.iloc[:quantidade]

    X_treino = amostra[colunas]
    y_treino = amostra["Is_laundering"].astype(int)

    preprocessador = ColumnTransformer([
        ("numericas", StandardScaler(), numericas_historico),
        (
            "categoricas",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categoricas
        )
    ], sparse_threshold=0)

    modelo = Pipeline([
        ("preprocessamento", preprocessador),
        (
            "classificador",
            HistGradientBoostingClassifier(
                max_iter=100,
                learning_rate=0.1,
                max_leaf_nodes=31,
                class_weight="balanced",
                early_stopping=False,
                random_state=42
            )
        )
    ])

    modelo.fit(X_treino, y_treino)

    # Desempenho nos próprios dados de treino
    prob_treino = modelo.predict_proba(X_treino)[:, 1]
    ap_treino = average_precision_score(y_treino, prob_treino)

    # Desempenho nos dados posteriores de validação
    prob_validacao = modelo.predict_proba(X_validacao)[:, 1]
    ap_validacao = average_precision_score(
        y_validacao, prob_validacao
    )

    resultados_curva.append({
        "Transações de treino": quantidade,
        "AP treino": ap_treino,
        "AP validação": ap_validacao
    })

# Mostrar os valores que formam o gráfico
curva = pd.DataFrame(resultados_curva)
display(curva.round(4))

# Gerar gráfico
plt.figure(figsize=(9, 5))

plt.plot(
    curva["Transações de treino"],
    curva["AP treino"],
    color="blue",
    marker="o",
    label="Treino"
)

plt.plot(
    curva["Transações de treino"],
    curva["AP validação"],
    color="red",
    marker="o",
    label="Validação"
)

plt.xlabel("Quantidade de transações de treino")
plt.ylabel("Average Precision (AP)")
plt.title("FraudRadar — Curva de aprendizado")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 1. Taxa de suspeitas por dia
print("=== DISTRIBUIÇÃO POR DATA ===")

display(
    df_exp.groupby("_data")["Is_laundering"]
    .agg(
        Transacoes="size",
        Suspeitas="sum",
        Taxa_suspeita="mean"
    )
    .assign(
        Taxa_suspeita=lambda x: (x["Taxa_suspeita"] * 100).round(3)
    )
)

# 2. Comportamento por tipo de pagamento
for nome, mascara in [
    ("Treino", treino_mask),
    ("Validação", validacao_mask)
]:
    print(f"\n=== PAYMENT_TYPE: {nome.upper()} ===")

    resultado = (
        df_exp.loc[mascara]
        .groupby("Payment_type")["Is_laundering"]
        .agg(
            Transacoes="size",
            Suspeitas="sum",
            Taxa_suspeita="mean"
        )
    )

    resultado["Taxa_suspeita"] *= 100

    display(resultado.round(3))

In [ ]:
import pandas as pd

caminho = r"C:\Users\Aluno\Downloads\archive\SAML-D.csv"

tipos = {
    "Time": "string",
    "Date": "string",
    "Sender_account": "int64",
    "Receiver_account": "int64",
    "Amount": "float64",
    "Payment_currency": "category",
    "Received_currency": "category",
    "Sender_bank_location": "category",
    "Receiver_bank_location": "category",
    "Payment_type": "category",
    "Is_laundering": "int8",
    "Laundering_type": "category"
}

df = pd.read_csv(
    caminho,
    dtype=tipos,
    low_memory=False
)

df["timestamp"] = pd.to_datetime(
    df["Date"] + " " + df["Time"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

print("Linhas e colunas:", df.shape)
print("Datas diferentes:", df["Date"].nunique())
print("Data inicial:", df["Date"].min())
print("Data final:", df["Date"].max())

print("\nDistribuição das classes:")
print(df["Is_laundering"].value_counts())

print("\nTaxa de suspeitas no dataset (%):")
print(round(df["Is_laundering"].mean() * 100, 4))

print("\nDatas inválidas:", df["timestamp"].isna().sum())
print("Ordenado cronologicamente:", df["timestamp"].is_monotonic_increasing)

memoria_gb = df.memory_usage(deep=True).sum() / 1024**3
print(f"Memória ocupada pelo DataFrame: {memoria_gb:.2f} GB")

In [ ]:
diario = df.groupby("Date")["Is_laundering"].agg(
    Transacoes="size",
    Suspeitas="sum",
    Taxa_suspeita="mean"
)

diario["Taxa_suspeita"] = (
    diario["Taxa_suspeita"] * 100
).round(3)

print("Primeiros cinco dias:")
display(diario.head())

print("Últimos cinco dias:")
display(diario.tail())

In [ ]:
import numpy as np
import pandas as pd

# Datas distintas em ordem cronológica
datas = np.sort(df["Date"].unique())

# Limites temporais: 70% treino, 15% validação e 15% teste
corte_treino = datas[int(len(datas) * 0.70)]
corte_teste = datas[int(len(datas) * 0.85)]

# Separação sem embaralhar os dados
treino_mask = df["Date"] < corte_treino

validacao_mask = (
    (df["Date"] >= corte_treino)
    & (df["Date"] < corte_teste)
)

teste_mask = df["Date"] >= corte_teste

# Conferir os três conjuntos
for nome, mascara in [
    ("TREINO", treino_mask),
    ("VALIDAÇÃO", validacao_mask),
    ("TESTE", teste_mask)
]:
    dados = df.loc[mascara, "Is_laundering"]

    print(f"\n=== {nome} ===")
    print("Período:", df.loc[mascara, "Date"].min(),
          "até", df.loc[mascara, "Date"].max())
    print("Transações:", len(dados))
    print("Suspeitas:", int(dados.sum()))
    print("Taxa de suspeitas (%):", round(dados.mean() * 100, 4))

In [ ]:
import numpy as np
import pandas as pd

# Preparar a base de trabalho, sem a coluna de tipologia
colunas = [
    "timestamp", "Date",
    "Sender_account", "Receiver_account", "Amount",
    "Payment_currency", "Received_currency",
    "Sender_bank_location", "Receiver_bank_location",
    "Payment_type", "Is_laundering"
]

df_feat = df[colunas].copy()
df_feat["_rowid"] = np.arange(len(df_feat), dtype=np.int32)

# 1. Histórico de envios por remetente e instante
print("Calculando histórico dos remetentes...")

envios = (
    df.groupby(["Sender_account", "timestamp"], sort=False)
    .agg(
        n_envios=("Amount", "size"),
        soma_valores=("Amount", "sum")
    )
    .reset_index()
    .sort_values(["Sender_account", "timestamp"])
)

g = envios.groupby("Sender_account", sort=False)

envios["envios_anteriores"] = (
    g["n_envios"].cumsum() - envios["n_envios"]
)

envios["soma_anterior"] = (
    g["soma_valores"].cumsum() - envios["soma_valores"]
)

envios["ultima_transacao"] = g["timestamp"].shift(1)

envios["media_valor_anterior"] = (
    envios["soma_anterior"]
    / envios["envios_anteriores"].replace(0, np.nan)
)

envios["segundos_desde_envio"] = (
    envios["timestamp"] - envios["ultima_transacao"]
).dt.total_seconds()

df_feat = df_feat.merge(
    envios[
        [
            "Sender_account", "timestamp",
            "envios_anteriores", "media_valor_anterior",
            "segundos_desde_envio"
        ]
    ],
    on=["Sender_account", "timestamp"],
    how="left",
    validate="many_to_one",
    sort=False
)

del envios, g

# 2. Histórico de recebimentos por destinatário
print("Calculando histórico dos destinatários...")

recebimentos = (
    df.groupby(["Receiver_account", "timestamp"], sort=False)
    .size()
    .rename("n_recebimentos")
    .reset_index()
    .sort_values(["Receiver_account", "timestamp"])
)

recebimentos["recebimentos_anteriores"] = (
    recebimentos.groupby("Receiver_account", sort=False)[
        "n_recebimentos"
    ].cumsum()
    - recebimentos["n_recebimentos"]
)

df_feat = df_feat.merge(
    recebimentos[
        ["Receiver_account", "timestamp", "recebimentos_anteriores"]
    ],
    on=["Receiver_account", "timestamp"],
    how="left",
    validate="many_to_one",
    sort=False
)

del recebimentos

# 3. Histórico da relação remetente-destinatário
print("Calculando histórico das relações entre contas...")

pares = (
    df.groupby(
        ["Sender_account", "Receiver_account", "timestamp"],
        sort=False
    )
    .size()
    .rename("n_transacoes_par")
    .reset_index()
    .sort_values(
        ["Sender_account", "Receiver_account", "timestamp"]
    )
)

pares["interacoes_anteriores"] = (
    pares.groupby(
        ["Sender_account", "Receiver_account"],
        sort=False
    )["n_transacoes_par"].cumsum()
    - pares["n_transacoes_par"]
)

df_feat = df_feat.merge(
    pares[
        [
            "Sender_account", "Receiver_account",
            "timestamp", "interacoes_anteriores"
        ]
    ],
    on=["Sender_account", "Receiver_account", "timestamp"],
    how="left",
    validate="many_to_one",
    sort=False
)

del pares

# 4. Restaurar a ordem original e tratar contas sem histórico
df_feat = (
    df_feat.sort_values("_rowid")
    .reset_index(drop=True)
)

df_feat["envios_anteriores"] = (
    df_feat["envios_anteriores"].fillna(0).astype("int32")
)

df_feat["recebimentos_anteriores"] = (
    df_feat["recebimentos_anteriores"].fillna(0).astype("int32")
)

df_feat["interacoes_anteriores"] = (
    df_feat["interacoes_anteriores"].fillna(0).astype("int32")
)

df_feat["media_valor_anterior"] = (
    df_feat["media_valor_anterior"].fillna(0).astype("float32")
)

df_feat["segundos_desde_envio"] = (
    df_feat["segundos_desde_envio"].fillna(-1).astype("float32")
)

# 5. Criar as características de data e valor
df_feat["hora"] = df_feat["timestamp"].dt.hour.astype("int8")
df_feat["dia_semana"] = df_feat["timestamp"].dt.dayofweek.astype("int8")
df_feat["valor_log"] = np.log1p(df_feat["Amount"]).astype("float32")

print("\nHistórico calculado!")
print("Dimensões:", df_feat.shape)
print(f"Memória utilizada: {df_feat.memory_usage(deep=True).sum() / 1024**3:.2f} GB")

display(
    df_feat[
        [
            "timestamp", "Sender_account", "Receiver_account",
            "Amount", "envios_anteriores", "media_valor_anterior",
            "segundos_desde_envio", "recebimentos_anteriores",
            "interacoes_anteriores", "Is_laundering"
        ]
    ].sample(10, random_state=42)
)

In [ ]:
import gc
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score

# 1. Definir as variáveis
numericas_base = ["valor_log", "hora", "dia_semana"]

numericas_historico = numericas_base + [
    "envios_anteriores",
    "media_valor_anterior",
    "segundos_desde_envio",
    "recebimentos_anteriores",
    "interacoes_anteriores"
]

categoricas = [
    "Payment_currency",
    "Received_currency",
    "Sender_bank_location",
    "Receiver_bank_location",
    "Payment_type"
]

# Reduzir o uso de memória das variáveis numéricas
for coluna in numericas_historico:
    df_feat[coluna] = df_feat[coluna].astype("float32")

# 2. Recriar a divisão temporal do dataset completo
datas = np.sort(df_feat["Date"].unique())

corte_treino = datas[int(len(datas) * 0.70)]
corte_teste = datas[int(len(datas) * 0.85)]

treino_mask = df_feat["Date"] < corte_treino

validacao_mask = (
    (df_feat["Date"] >= corte_treino)
    & (df_feat["Date"] < corte_teste)
)

teste_mask = df_feat["Date"] >= corte_teste

y_treino = df_feat.loc[treino_mask, "Is_laundering"].astype("int8")
y_validacao = df_feat.loc[validacao_mask, "Is_laundering"].astype("int8")
y_teste = df_feat.loc[teste_mask, "Is_laundering"].astype("int8")

print("Treino:", len(y_treino), "| suspeitas:", int(y_treino.sum()))
print("Validação:", len(y_validacao), "| suspeitas:", int(y_validacao.sum()))
print("Teste:", len(y_teste), "| suspeitas:", int(y_teste.sum()))

# 3. Configurar os experimentos
experimentos = [
    ("Logística - sem histórico", numericas_base, "logistica"),
    ("Logística - com histórico", numericas_historico, "logistica"),
    ("HistGradientBoosting - sem histórico", numericas_base, "hgb"),
    ("HistGradientBoosting - com histórico", numericas_historico, "hgb")
]

resultados_modelos = []
resultados_topk = []
ks = [300, 1000, 3000, 5000]

# 4. Treinar cada modelo individualmente
for nome, numericas, algoritmo in experimentos:

    print(f"\nTreinando: {nome}...", flush=True)

    colunas = numericas + categoricas

    X_tr = df_feat.loc[treino_mask, colunas]
    X_val = df_feat.loc[validacao_mask, colunas]
    X_te = df_feat.loc[teste_mask, colunas]

    if algoritmo == "logistica":
        # Matriz esparsa para economizar memória
        preprocessador = ColumnTransformer([
            ("num", StandardScaler(), numericas),
            ("cat", OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True,
                dtype=np.float32
            ), categoricas)
        ], sparse_threshold=1.0)

        classificador = LogisticRegression(
            max_iter=200,
            tol=1e-3,
            class_weight="balanced",
            random_state=42
        )

    else:
        # HistGradientBoosting precisa de matriz densa
        preprocessador = ColumnTransformer([
            ("num", "passthrough", numericas),
            ("cat", OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
                dtype=np.float32
            ), categoricas)
        ], sparse_threshold=0)

        classificador = HistGradientBoostingClassifier(
            max_iter=100,
            learning_rate=0.1,
            max_leaf_nodes=31,
            class_weight="balanced",
            early_stopping=False,
            random_state=42
        )

    modelo = Pipeline([
        ("preprocessamento", preprocessador),
        ("classificador", classificador)
    ])

    # Ajustar exclusivamente no treino
    modelo.fit(X_tr, y_treino)

    # Avaliar a ordenação na validação e no teste
    prob_val = modelo.predict_proba(X_val)[:, 1]
    prob_teste = modelo.predict_proba(X_te)[:, 1]

    ap_val = average_precision_score(y_validacao, prob_val)
    ap_teste = average_precision_score(y_teste, prob_teste)

    resultados_modelos.append({
        "Modelo": nome,
        "AP validação": ap_val,
        "AP teste": ap_teste,
        "Taxa suspeita teste (%)": y_teste.mean() * 100
    })

    # Avaliar as transações mais bem classificadas
    y_test_array = y_teste.to_numpy()
    max_k = min(max(ks), len(prob_teste))

    indices_top = np.argpartition(
        prob_teste, -max_k
    )[-max_k:]

    indices_top = indices_top[
        np.argsort(prob_teste[indices_top])[::-1]
    ]

    for k in ks:
        indices = indices_top[:min(k, len(indices_top))]
        encontrados = int(y_test_array[indices].sum())

        resultados_topk.append({
            "Modelo": nome,
            "Top K transações": len(indices),
            "Suspeitas encontradas": encontrados,
            "Precisão (%)": encontrados / len(indices) * 100,
            "Recall (%)": encontrados / int(y_teste.sum()) * 100
        })

    print(f"AP validação: {ap_val:.4f}")
    print(f"AP teste: {ap_teste:.4f}")
    print("Treinamento concluído.", flush=True)

    del modelo, preprocessador, classificador
    del X_tr, X_val, X_te, prob_val, prob_teste, indices_top
    gc.collect()

# 5. Mostrar as comparações
comparacao = pd.DataFrame(resultados_modelos)
fila_alertas = pd.DataFrame(resultados_topk)

print("\n=== COMPARAÇÃO DOS MODELOS ===")
display(comparacao.round(4))

print("\n=== DESEMPENHO NAS FILAS DE ALERTAS ===")
display(fila_alertas.round(3))

In [ ]:
# Auditar os históricos de algumas transações do teste

import numpy as np
import pandas as pd

# Selecionar 3 suspeitas e 3 normais do teste
dados_teste = df_feat.loc[teste_mask]

amostra_suspeitas = dados_teste[
    dados_teste["Is_laundering"] == 1
].sample(3, random_state=42)

amostra_normais = dados_teste[
    dados_teste["Is_laundering"] == 0
].sample(3, random_state=42)

amostra = pd.concat([amostra_suspeitas, amostra_normais])

verificacoes = []

for _, r in amostra.iterrows():
    t = r["timestamp"]
    remetente = r["Sender_account"]
    destinatario = r["Receiver_account"]

    # Somente transações estritamente anteriores
    anteriores_envio = df.loc[
        (df["Sender_account"] == remetente)
        & (df["timestamp"] < t),
        ["timestamp", "Amount"]
    ]

    anteriores_recebimento = df.loc[
        (df["Receiver_account"] == destinatario)
        & (df["timestamp"] < t)
    ]

    anteriores_par = df.loc[
        (df["Sender_account"] == remetente)
        & (df["Receiver_account"] == destinatario)
        & (df["timestamp"] < t)
    ]

    # Valores esperados, calculados diretamente
    qtd_envios = len(anteriores_envio)

    media_envios = (
        anteriores_envio["Amount"].mean()
        if qtd_envios else 0
    )

    segundos_desde = (
        (t - anteriores_envio["timestamp"].max()).total_seconds()
        if qtd_envios else -1
    )

    qtd_recebimentos = len(anteriores_recebimento)
    qtd_par = len(anteriores_par)

    esperados = {
        "envios_anteriores": qtd_envios,
        "media_valor_anterior": media_envios,
        "segundos_desde_envio": segundos_desde,
        "recebimentos_anteriores": qtd_recebimentos,
        "interacoes_anteriores": qtd_par
    }

    for coluna, esperado in esperados.items():
        calculado = r[coluna]

        confere = np.isclose(
            calculado,
            esperado,
            rtol=1e-4,
            atol=0.05
        )

        verificacoes.append({
            "Classe": int(r["Is_laundering"]),
            "Data/hora": t,
            "Variável": coluna,
            "Valor armazenado": calculado,
            "Valor recalculado": esperado,
            "Confere": confere
        })

auditoria = pd.DataFrame(verificacoes)

display(auditoria)

print(
    "\nTodas as verificações passaram:",
    auditoria["Confere"].all()
)

In [ ]:
print("Colunas utilizadas pelo modelo com histórico:")
print(numericas_historico + categoricas)

print("\nColunas que não devem entrar no modelo:")
print([
    "Is_laundering",
    "Laundering_type",
    "Sender_account",
    "Receiver_account"
])

In [ ]:
import gc
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score

# Grupos de variáveis para o teste de ablação
grupos = {
    "A - Sem histórico": numericas_base,

    "B - Contagens": numericas_base + [
        "envios_anteriores",
        "recebimentos_anteriores",
        "interacoes_anteriores"
    ],

    "C - Comportamento": numericas_base + [
        "media_valor_anterior",
        "segundos_desde_envio"
    ],

    "D - Histórico completo": numericas_historico
}

categoricas = [
    "Payment_currency",
    "Received_currency",
    "Sender_bank_location",
    "Receiver_bank_location",
    "Payment_type"
]

y_val = df_feat.loc[validacao_mask, "Is_laundering"].astype("int8")
y_test = df_feat.loc[teste_mask, "Is_laundering"].astype("int8")

resultados = []

for nome, numericas in grupos.items():

    print(f"Treinando {nome}...", flush=True)

    colunas = numericas + categoricas

    X_tr = df_feat.loc[treino_mask, colunas]
    X_val = df_feat.loc[validacao_mask, colunas]
    X_test = df_feat.loc[teste_mask, colunas]

    preprocessador = ColumnTransformer([
        ("num", "passthrough", numericas),
        ("cat", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
            dtype=np.float32
        ), categoricas)
    ], sparse_threshold=0)

    modelo = Pipeline([
        ("preprocessamento", preprocessador),
        ("classificador", HistGradientBoostingClassifier(
            max_iter=100,
            learning_rate=0.1,
            max_leaf_nodes=31,
            class_weight="balanced",
            early_stopping=False,
            random_state=42
        ))
    ])

    modelo.fit(
        X_tr,
        df_feat.loc[treino_mask, "Is_laundering"].astype("int8")
    )

    prob_val = modelo.predict_proba(X_val)[:, 1]
    prob_test = modelo.predict_proba(X_test)[:, 1]

    # Desempenho por ordenação de risco
    ap_val = average_precision_score(y_val, prob_val)
    ap_test = average_precision_score(y_test, prob_test)

    # Quantas suspeitas estão nas 1.000 maiores pontuações?
    k = min(1000, len(prob_test))
    top = np.argpartition(prob_test, -k)[-k:]
    suspeitas_top = int(y_test.to_numpy()[top].sum())

    resultados.append({
        "Experimento": nome,
        "AP validação": ap_val,
        "AP teste": ap_test,
        "Suspeitas no Top 1000": suspeitas_top,
        "Precisão Top 1000 (%)": suspeitas_top / k * 100
    })

    del modelo, preprocessador, X_tr, X_val, X_test
    del prob_val, prob_test, top
    gc.collect()

display(pd.DataFrame(resultados).round(4))

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score
import numpy as np
import pandas as pd
import gc

# Histórico testado individualmente
variantes = {
    "Base + envios anteriores":
        numericas_base + ["envios_anteriores"],

    "Base + recebimentos anteriores":
        numericas_base + ["recebimentos_anteriores"],

    "Base + interações anteriores":
        numericas_base + ["interacoes_anteriores"]
}

y_tr = df_feat.loc[treino_mask, "Is_laundering"].astype("int8")
y_val = df_feat.loc[validacao_mask, "Is_laundering"].astype("int8")
y_test = df_feat.loc[teste_mask, "Is_laundering"].astype("int8")

resultados_individuais = []

for nome, numericas in variantes.items():

    print(f"Treinando: {nome}...", flush=True)

    colunas = numericas + categoricas

    X_tr = df_feat.loc[treino_mask, colunas]
    X_val = df_feat.loc[validacao_mask, colunas]
    X_test = df_feat.loc[teste_mask, colunas]

    preprocessador = ColumnTransformer([
        ("num", "passthrough", numericas),
        ("cat", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
            dtype=np.float32
        ), categoricas)
    ], sparse_threshold=0)

    modelo = Pipeline([
        ("preprocessamento", preprocessador),
        ("classificador", HistGradientBoostingClassifier(
            max_iter=100,
            learning_rate=0.1,
            max_leaf_nodes=31,
            class_weight="balanced",
            early_stopping=False,
            random_state=42
        ))
    ])

    modelo.fit(X_tr, y_tr)

    prob_val = modelo.predict_proba(X_val)[:, 1]
    prob_test = modelo.predict_proba(X_test)[:, 1]

    # Métricas de ordenação
    ap_val = average_precision_score(y_val, prob_val)
    ap_test = average_precision_score(y_test, prob_test)

    # Suspeitas entre as 1.000 maiores pontuações do teste
    k = min(1000, len(prob_test))
    top = np.argpartition(prob_test, -k)[-k:]
    suspeitas_top = int(y_test.to_numpy()[top].sum())

    resultados_individuais.append({
        "Experimento": nome,
        "AP validação": ap_val,
        "AP teste": ap_test,
        "Suspeitas no Top 1000": suspeitas_top,
        "Precisão Top 1000 (%)": 100 * suspeitas_top / k
    })

    print(
        f"AP validação: {ap_val:.4f} | "
        f"AP teste: {ap_test:.4f} | "
        f"Suspeitas no Top 1000: {suspeitas_top}",
        flush=True
    )

    del modelo, preprocessador, X_tr, X_val, X_test
    del prob_val, prob_test, top
    gc.collect()

display(pd.DataFrame(resultados_individuais).round(4))

In [ ]:
import gc
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score

# Contagens históricas
envios = "envios_anteriores"
recebimentos = "recebimentos_anteriores"
interacoes = "interacoes_anteriores"

# Experimentos: nome, variáveis numéricas, categóricas
experimentos = [
    (
        "Somente as 3 contagens",
        [envios, recebimentos, interacoes],
        []
    ),
    (
        "Base + envios e recebimentos",
        numericas_base + [envios, recebimentos],
        categoricas
    ),
    (
        "Base + envios e interações",
        numericas_base + [envios, interacoes],
        categoricas
    ),
    (
        "Base + recebimentos e interações",
        numericas_base + [recebimentos, interacoes],
        categoricas
    ),
    (
        "Base + as 3 contagens",
        numericas_base + [envios, recebimentos, interacoes],
        categoricas
    )
]

y_tr = df_feat.loc[treino_mask, "Is_laundering"].astype("int8")
y_val = df_feat.loc[validacao_mask, "Is_laundering"].astype("int8")
y_test = df_feat.loc[teste_mask, "Is_laundering"].astype("int8")

resultados = []

for nome, numericas, categorias in experimentos:
    print(f"Treinando: {nome}...", flush=True)

    colunas = numericas + categorias

    X_tr = df_feat.loc[treino_mask, colunas]
    X_val = df_feat.loc[validacao_mask, colunas]
    X_test = df_feat.loc[teste_mask, colunas]

    transformacoes = [
        ("num", "passthrough", numericas)
    ]

    if categorias:
        transformacoes.append(
            (
                "cat",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                    dtype=np.float32
                ),
                categorias
            )
        )

    preprocessador = ColumnTransformer(
        transformers=transformacoes,
        sparse_threshold=0
    )

    modelo = Pipeline([
        ("preprocessamento", preprocessador),
        (
            "classificador",
            HistGradientBoostingClassifier(
                max_iter=100,
                learning_rate=0.1,
                max_leaf_nodes=31,
                class_weight="balanced",
                early_stopping=False,
                random_state=42
            )
        )
    ])

    modelo.fit(X_tr, y_tr)

    prob_val = modelo.predict_proba(X_val)[:, 1]
    prob_test = modelo.predict_proba(X_test)[:, 1]

    # Métricas calculadas sem escolher um limiar
    ap_val = average_precision_score(y_val, prob_val)
    ap_test = average_precision_score(y_test, prob_test)

    # Quantidade de suspeitas entre as mil maiores pontuações
    k = 1000
    top = np.argpartition(prob_test, -k)[-k:]
    suspeitas_top = int(y_test.to_numpy()[top].sum())

    resultados.append({
        "Experimento": nome,
        "AP validação": ap_val,
        "AP teste": ap_test,
        "Suspeitas no Top 1000": suspeitas_top,
        "Precisão Top 1000 (%)": suspeitas_top / k * 100
    })

    print(
        f"AP validação: {ap_val:.4f} | "
        f"AP teste: {ap_test:.4f} | "
        f"Suspeitas Top 1000: {suspeitas_top}",
        flush=True
    )

    del modelo, preprocessador, X_tr, X_val, X_test
    del prob_val, prob_test, top
    gc.collect()

display(pd.DataFrame(resultados).round(4))

In [ ]:
import numpy as np
import pandas as pd

# Analisar os conjuntos posteriores ao treino
mask = validacao_mask | teste_mask

auditoria = df_feat.loc[
    mask,
    [
        "envios_anteriores",
        "recebimentos_anteriores",
        "interacoes_anteriores",
        "Is_laundering"
    ]
].copy()

auditoria["conjunto"] = np.where(
    validacao_mask.loc[mask],
    "Validacao",
    "Teste"
)

# Agrupar as contagens em faixas
faixas = [-1, 0, 1, 5, 10, 25, 50, 100, np.inf]

rotulos = [
    "0", "1", "2-5", "6-10",
    "11-25", "26-50", "51-100", ">100"
]

colunas_contagem = [
    "envios_anteriores",
    "recebimentos_anteriores",
    "interacoes_anteriores"
]

for coluna in colunas_contagem:
    auditoria[coluna + "_faixa"] = pd.cut(
        auditoria[coluna],
        bins=faixas,
        labels=rotulos
    )

# Calcular a quantidade e a taxa de suspeitas por combinação
resumo = (
    auditoria.groupby(
        [
            "conjunto",
            "envios_anteriores_faixa",
            "recebimentos_anteriores_faixa",
            "interacoes_anteriores_faixa"
        ],
        observed=True
    )["Is_laundering"]
    .agg(
        transacoes="size",
        suspeitas="sum"
    )
    .reset_index()
)

resumo["taxa_suspeita_%"] = (
    resumo["suspeitas"] / resumo["transacoes"] * 100
).round(3)

# Exibir os padrões mais suspeitos com pelo menos 100 registros
for periodo in ["Validacao", "Teste"]:
    print(f"\n=== COMBINAÇÕES NO {periodo.upper()} ===")

    resultado = resumo[
        (resumo["conjunto"] == periodo)
        & (resumo["transacoes"] >= 100)
    ].sort_values(
        ["taxa_suspeita_%", "transacoes"],
        ascending=[False, False]
    )

    display(resultado.head(20))

del auditoria

In [ ]:
import pandas as pd

regras = {
    "Remetente >100, destinatário 11-25, par novo":
        lambda d: (
            (d["envios_anteriores"] > 100)
            & d["recebimentos_anteriores"].between(11, 25)
            & (d["interacoes_anteriores"] == 0)
        ),

    "Remetente >100, destinatário 26-50, par novo":
        lambda d: (
            (d["envios_anteriores"] > 100)
            & d["recebimentos_anteriores"].between(26, 50)
            & (d["interacoes_anteriores"] == 0)
        ),

    "Remetente >100, destinatário >100, par novo":
        lambda d: (
            (d["envios_anteriores"] > 100)
            & (d["recebimentos_anteriores"] > 100)
            & (d["interacoes_anteriores"] == 0)
        )
}

for periodo, mascara in [
    ("VALIDAÇÃO", validacao_mask),
    ("TESTE", teste_mask)
]:
    print(f"\n{'=' * 60}")
    print(periodo)
    print("=" * 60)

    parte = df_feat.loc[
        mascara,
        [
            "envios_anteriores",
            "recebimentos_anteriores",
            "interacoes_anteriores",
            "Is_laundering"
        ]
    ].copy()

    # A tipologia é consultada apenas para investigação.
    parte["Laundering_type"] = (
        df.loc[mascara, "Laundering_type"].to_numpy()
    )

    for nome, regra in regras.items():
        selecionadas = parte.loc[regra(parte)]

        if selecionadas.empty:
            continue

        print(f"\n--- {nome} ---")
        print("Transações:", len(selecionadas))
        print(
            "Suspeitas:",
            int(selecionadas["Is_laundering"].sum())
        )
        print(
            "Taxa de suspeitas (%):",
            round(
                selecionadas["Is_laundering"].mean() * 100, 3
            )
        )

        print("\nTipologias encontradas:")
        display(
            selecionadas["Laundering_type"]
            .value_counts()
            .head(10)
            .to_frame("Quantidade")
        )

In [ ]:
import pandas as pd

# Três regras candidatas identificadas na análise exploratória
regras = {
    "A: >100 envios, 11-25 recebimentos, par novo": (
        (df_feat["envios_anteriores"] > 100)
        & df_feat["recebimentos_anteriores"].between(11, 25)
        & (df_feat["interacoes_anteriores"] == 0)
    ),

    "B: >100 envios, 26-50 recebimentos, par novo": (
        (df_feat["envios_anteriores"] > 100)
        & df_feat["recebimentos_anteriores"].between(26, 50)
        & (df_feat["interacoes_anteriores"] == 0)
    ),

    "C: >100 envios, >100 recebimentos, par novo": (
        (df_feat["envios_anteriores"] > 100)
        & (df_feat["recebimentos_anteriores"] > 100)
        & (df_feat["interacoes_anteriores"] == 0)
    )
}

periodos = {
    "Treino": treino_mask,
    "Validação": validacao_mask,
    "Teste": teste_mask
}

resultados = []

for nome_regra, regra in regras.items():

    for nome_periodo, periodo in periodos.items():

        # Transações sinalizadas por essa regra
        selecionadas = regra & periodo

        total_alertas = int(selecionadas.sum())

        suspeitas = int(
            df_feat.loc[selecionadas, "Is_laundering"].sum()
        )

        total_suspeitas = int(
            df_feat.loc[periodo, "Is_laundering"].sum()
        )

        resultados.append({
            "Regra": nome_regra,
            "Período": nome_periodo,
            "Transações sinalizadas": total_alertas,
            "Suspeitas encontradas": suspeitas,
            "Falsos positivos": total_alertas - suspeitas,
            "Precisão (%)": (
                100 * suspeitas / total_alertas
                if total_alertas else 0
            ),
            "Cobertura das suspeitas (%)": (
                100 * suspeitas / total_suspeitas
                if total_suspeitas else 0
            )
        })

display(
    pd.DataFrame(resultados).round(3)
)

In [ ]:
import numpy as np
import pandas as pd

# As três regras identificadas
regra_a = (
    (df_feat["envios_anteriores"] > 100)
    & df_feat["recebimentos_anteriores"].between(11, 25)
    & (df_feat["interacoes_anteriores"] == 0)
)

regra_b = (
    (df_feat["envios_anteriores"] > 100)
    & df_feat["recebimentos_anteriores"].between(26, 50)
    & (df_feat["interacoes_anteriores"] == 0)
)

regra_c = (
    (df_feat["envios_anteriores"] > 100)
    & (df_feat["recebimentos_anteriores"] > 100)
    & (df_feat["interacoes_anteriores"] == 0)
)

# Combinar as regras
alerta = (regra_a | regra_b | regra_c).astype("int8")
y = df_feat["Is_laundering"].astype("int8")

# Montar a base de análise
auditoria = pd.DataFrame({
    "Data": pd.to_datetime(df_feat["Date"]),
    "y": y,
    "alerta": alerta
})

auditoria["Conjunto"] = np.select(
    [
        treino_mask.to_numpy(),
        validacao_mask.to_numpy(),
        teste_mask.to_numpy()
    ],
    ["Treino", "Validação", "Teste"],
    default="Outro"
)

auditoria["Mês"] = auditoria["Data"].dt.to_period("M").astype(str)

auditoria["VP"] = (
    (auditoria["alerta"] == 1) & (auditoria["y"] == 1)
).astype("int8")

auditoria["FP"] = (
    (auditoria["alerta"] == 1) & (auditoria["y"] == 0)
).astype("int8")

# 1. Desempenho agregado por conjunto
resumo = auditoria.groupby("Conjunto").agg(
    Transacoes=("y", "size"),
    Suspeitas_reais=("y", "sum"),
    Alertas=("alerta", "sum"),
    Verdadeiros_positivos=("VP", "sum"),
    Falsos_positivos=("FP", "sum")
)

resumo["Precisao_%"] = (
    100 * resumo["Verdadeiros_positivos"]
    / resumo["Alertas"].replace(0, np.nan)
)

resumo["Cobertura_%"] = (
    100 * resumo["Verdadeiros_positivos"]
    / resumo["Suspeitas_reais"].replace(0, np.nan)
)

print("=== DESEMPENHO AGREGADO ===")
display(resumo.round(3))

# 2. Desempenho mês a mês
mensal = auditoria.groupby(["Conjunto", "Mês"]).agg(
    Transacoes=("y", "size"),
    Suspeitas_reais=("y", "sum"),
    Alertas=("alerta", "sum"),
    Verdadeiros_positivos=("VP", "sum"),
    Falsos_positivos=("FP", "sum")
).reset_index()

mensal["Precisao_%"] = (
    100 * mensal["Verdadeiros_positivos"]
    / mensal["Alertas"].replace(0, np.nan)
)

mensal["Cobertura_%"] = (
    100 * mensal["Verdadeiros_positivos"]
    / mensal["Suspeitas_reais"].replace(0, np.nan)
)

print("=== RESULTADOS MENSAIS ===")
display(
    mensal.loc[
        mensal["Conjunto"].isin(["Treino", "Validação", "Teste"])
    ].round(3)
)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score

# 1. Treinar o modelo com histórico
colunas = numericas_historico + categoricas

X_tr = df_feat.loc[treino_mask, colunas]
X_te = df_feat.loc[teste_mask, colunas]

y_tr = df_feat.loc[treino_mask, "Is_laundering"].astype("int8")
y_te = df_feat.loc[teste_mask, "Is_laundering"].astype("int8")

preprocessador = ColumnTransformer([
    ("num", "passthrough", numericas_historico),
    ("cat", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
        dtype=np.float32
    ), categoricas)
], sparse_threshold=0)

modelo = Pipeline([
    ("preprocessamento", preprocessador),
    ("classificador", HistGradientBoostingClassifier(
        max_iter=100,
        learning_rate=0.1,
        max_leaf_nodes=31,
        class_weight="balanced",
        early_stopping=False,
        random_state=42
    ))
])

print("Treinando o modelo...")
modelo.fit(X_tr, y_tr)

scores = modelo.predict_proba(X_te)[:, 1]

# 2. Aplicar as três regras no conjunto de teste
teste = df_feat.loc[teste_mask]

regra_a = (
    (teste["envios_anteriores"] > 100)
    & teste["recebimentos_anteriores"].between(11, 25)
    & (teste["interacoes_anteriores"] == 0)
)

regra_b = (
    (teste["envios_anteriores"] > 100)
    & teste["recebimentos_anteriores"].between(26, 50)
    & (teste["interacoes_anteriores"] == 0)
)

regra_c = (
    (teste["envios_anteriores"] > 100)
    & (teste["recebimentos_anteriores"] > 100)
    & (teste["interacoes_anteriores"] == 0)
)

alertas_regras = (regra_a | regra_b | regra_c).to_numpy()
y_real = y_te.to_numpy()

# 3. Estratégias de seleção
ordem = np.argsort(scores)[::-1]

# Somente regras
alertas_apenas_regras = alertas_regras.copy()

# ML, com o mesmo orçamento de 438 alertas observado nas regras
k_regras = int(alertas_regras.sum())
alertas_ml_438 = np.zeros(len(y_real), dtype=bool)
alertas_ml_438[ordem[:k_regras]] = True

# ML, orçamento de 1.000 alertas
k_total = min(1000, len(y_real))
alertas_ml_1000 = np.zeros(len(y_real), dtype=bool)
alertas_ml_1000[ordem[:k_total]] = True

# Híbrida: primeiro as regras, depois completar até 1.000 alertas
alertas_hibridos = alertas_regras.copy()
vagas = max(0, k_total - int(alertas_hibridos.sum()))

candidatos = ordem[~alertas_hibridos[ordem]]
alertas_hibridos[candidatos[:vagas]] = True

# 4. Métricas comparáveis
def avaliar(nome, alertas):
    tp = int((alertas & (y_real == 1)).sum())
    fp = int((alertas & (y_real == 0)).sum())
    fn = int((~alertas & (y_real == 1)).sum())

    return {
        "Estratégia": nome,
        "Alertas": int(alertas.sum()),
        "Suspeitas encontradas": tp,
        "Falsos positivos": fp,
        "Precisão (%)": round(
            100 * tp / alertas.sum(), 3
        ) if alertas.sum() else 0,
        "Cobertura (%)": round(
            100 * tp / (tp + fn), 3
        ) if tp + fn else 0
    }

resultados = [
    avaliar("Somente regras", alertas_apenas_regras),
    avaliar("ML — 438 alertas", alertas_ml_438),
    avaliar("ML — 1.000 alertas", alertas_ml_1000),
    avaliar("Híbrida — 1.000 alertas", alertas_hibridos)
]

display(pd.DataFrame(resultados))

print(
    "\nAverage Precision do ML:",
    round(average_precision_score(y_real, scores), 4)
)

In [ ]:
import numpy as np
import pandas as pd

# Dados do período de teste, na mesma ordem das previsões
teste = df_feat.loc[teste_mask]

y_real = teste["Is_laundering"].to_numpy()

# Regras históricas
regra_a = (
    (teste["envios_anteriores"] > 100)
    & teste["recebimentos_anteriores"].between(11, 25)
    & (teste["interacoes_anteriores"] == 0)
)

regra_b = (
    (teste["envios_anteriores"] > 100)
    & teste["recebimentos_anteriores"].between(26, 50)
    & (teste["interacoes_anteriores"] == 0)
)

regra_c = (
    (teste["envios_anteriores"] > 100)
    & (teste["recebimentos_anteriores"] > 100)
    & (teste["interacoes_anteriores"] == 0)
)

alertas_regras = (regra_a | regra_b | regra_c).to_numpy()

# Reutilizar as pontuações calculadas pelo modelo
# 'scores' deve corresponder às linhas de X_te do conjunto de teste
assert len(scores) == len(teste), "As previsões não correspondem ao teste."

k = 1000
indices_top = np.argsort(scores)[-k:]

alertas_modelo = np.zeros(len(teste), dtype=bool)
alertas_modelo[indices_top] = True

# Comparar a sobreposição
intersecao = alertas_regras & alertas_modelo
somente_regras = alertas_regras & ~alertas_modelo
somente_modelo = alertas_modelo & ~alertas_regras

resumo = pd.DataFrame([
    {
        "Grupo": "Regras e modelo",
        "Transações": int(intersecao.sum()),
        "Suspeitas": int(y_real[intersecao].sum())
    },
    {
        "Grupo": "Somente regras",
        "Transações": int(somente_regras.sum()),
        "Suspeitas": int(y_real[somente_regras].sum())
    },
    {
        "Grupo": "Somente modelo",
        "Transações": int(somente_modelo.sum()),
        "Suspeitas": int(y_real[somente_modelo].sum())
    }
])

display(resumo)

In [ ]:
import numpy as np
import pandas as pd

# Dados correspondentes ao período de teste
teste = df_feat.loc[teste_mask].copy()

# Confirmar que as pontuações correspondem às transações
assert len(scores) == len(teste), (
    "As pontuações não correspondem às linhas do conjunto de teste."
)

# Score e posição no ranking
teste["score_risco"] = scores
teste["posicao"] = (
    teste["score_risco"]
    .rank(method="first", ascending=False)
    .astype(int)
)

# Regras históricas
regra_a = (
    (teste["envios_anteriores"] > 100)
    & teste["recebimentos_anteriores"].between(11, 25)
    & (teste["interacoes_anteriores"] == 0)
)

regra_b = (
    (teste["envios_anteriores"] > 100)
    & teste["recebimentos_anteriores"].between(26, 50)
    & (teste["interacoes_anteriores"] == 0)
)

regra_c = (
    (teste["envios_anteriores"] > 100)
    & (teste["recebimentos_anteriores"] > 100)
    & (teste["interacoes_anteriores"] == 0)
)

teste["regra_a"] = regra_a
teste["regra_b"] = regra_b
teste["regra_c"] = regra_c

# Justificativa legível para a fila
teste["motivo"] = np.select(
    [regra_a, regra_b, regra_c],
    [
        "Mais de 100 envios anteriores; "
        "11 a 25 recebimentos anteriores do destinatário; "
        "nenhuma interação anterior entre o par.",

        "Mais de 100 envios anteriores; "
        "26 a 50 recebimentos anteriores do destinatário; "
        "nenhuma interação anterior entre o par.",

        "Mais de 100 envios anteriores; "
        "mais de 100 recebimentos anteriores do destinatário; "
        "nenhuma interação anterior entre o par."
    ],
    default="Score elevado do modelo; sem regra explícita correspondente."
)

# Selecionar os 1.000 alertas de maior risco
fila_alertas = (
    teste.loc[teste["posicao"] <= 1000, [
        "posicao",
        "timestamp",
        "Amount",
        "Payment_type",
        "score_risco",
        "envios_anteriores",
        "recebimentos_anteriores",
        "interacoes_anteriores",
        "motivo"
    ]]
    .sort_values("posicao")
    .reset_index(drop=True)
)

# Mostrar uma amostra da fila do analista
display(fila_alertas.head(30))

print("\nTotal de alertas na fila:", len(fila_alertas))

print(
    "Alertas com regra explícita:",
    int((fila_alertas["regra_a"] if "regra_a" in fila_alertas.columns
         else pd.Series(dtype=bool)).sum())
    if False else int(
        (
            teste.loc[teste["posicao"] <= 1000, "regra_a"]
            | teste.loc[teste["posicao"] <= 1000, "regra_b"]
            | teste.loc[teste["posicao"] <= 1000, "regra_c"]
        ).sum()
    )
)

# Exportar a fila para inspeção no Excel
fila_alertas.to_csv(
    "fraudradar_fila_alertas.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Arquivo salvo: fraudradar_fila_alertas.csv")

In [ ]:
import numpy as np
import pandas as pd

# Recriar a base do período de teste, na ordem das previsões
teste = df_feat.loc[teste_mask].copy()

assert len(scores) == len(teste), (
    "As pontuações não correspondem às transações do teste."
)

teste["score_risco"] = scores

# Posição no ranking: 1 = maior pontuação
teste["posicao"] = (
    teste["score_risco"]
    .rank(method="first", ascending=False)
    .astype(int)
)

# Regras históricas
regra_a = (
    (teste["envios_anteriores"] > 100)
    & teste["recebimentos_anteriores"].between(11, 25)
    & (teste["interacoes_anteriores"] == 0)
)

regra_b = (
    (teste["envios_anteriores"] > 100)
    & teste["recebimentos_anteriores"].between(26, 50)
    & (teste["interacoes_anteriores"] == 0)
)

regra_c = (
    (teste["envios_anteriores"] > 100)
    & (teste["recebimentos_anteriores"] > 100)
    & (teste["interacoes_anteriores"] == 0)
)

# Identificar a origem da sinalização
teste["regra_acionada"] = np.select(
    [regra_a, regra_b, regra_c],
    ["Regra A", "Regra B", "Regra C"],
    default="Modelo ML"
)

teste["motivo"] = np.select(
    [regra_a, regra_b, regra_c],
    [
        "Mais de 100 envios anteriores; "
        "11 a 25 recebimentos anteriores do destinatário; "
        "nenhuma interação anterior entre o par.",

        "Mais de 100 envios anteriores; "
        "26 a 50 recebimentos anteriores do destinatário; "
        "nenhuma interação anterior entre o par.",

        "Mais de 100 envios anteriores; "
        "mais de 100 recebimentos anteriores do destinatário; "
        "nenhuma interação anterior entre o par."
    ],
    default=(
        "Prioridade atribuída pelo modelo de machine learning; "
        "nenhuma das três regras explícitas foi acionada."
    )
)

# Construir a fila dos mil alertas prioritários
colunas_fila = [
    "posicao",
    "timestamp",
    "Sender_account",
    "Receiver_account",
    "Amount",
    "Payment_type",
    "score_risco",
    "envios_anteriores",
    "media_valor_anterior",
    "segundos_desde_envio",
    "recebimentos_anteriores",
    "interacoes_anteriores",
    "regra_acionada",
    "motivo"
]

fila_alertas = (
    teste.loc[teste["posicao"] <= 1000, colunas_fila]
    .sort_values("posicao")
    .reset_index(drop=True)
)

# Conferir a fila
print("Total de alertas:", len(fila_alertas))
print("\nOrigem dos alertas:")
display(
    fila_alertas["regra_acionada"]
    .value_counts()
    .to_frame("Quantidade")
)

display(fila_alertas.head(10))

# Exportar para Excel/CSV
fila_alertas.to_csv(
    "fraudradar_fila_alertas.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nArquivo atualizado: fraudradar_fila_alertas.csv")